In [1]:
import pandas

/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('Detailed_Polling_Data.csv')

<IPython.core.display.Javascript object>

In [3]:
df.columns

Index(['Serial No. Of Polling Station',
       'All India Anna Dravida Munnetra Kazhagam', 'Naam Tamilar Katchi',
       'Desiya Murpokku Dravida Kazhagam', 'Tamilaga Vettri Kazhagam',
       'Puthiya Tamilagam', 'Nam Naadu Nam Makkal Nam Ethirkaalam Katchi',
       'Desa Makkal Munnetrak Kazhagam', 'Independent', 'Independent.1',
       'Independent.2', 'Independent.3', 'Independent.4', 'Independent.5',
       'Independent.6', 'Independent.7', 'Total of Valid Votes',
       'No. Of Rejected Votes', 'NOTA', 'TotalNo. Of Tendered Votes',
       'Location and name of Building in Which Polling Station Located',
       'Polling Area', 'Polling station Type', 'Winner_Party', 'Winner_Votes',
       'Runner_Up_Votes', 'Margin_Of_Victory', 'Runner_Up_Party',
       'All India Anna Dravida Munnetra Kazhagam_Rank',
       'Naam Tamilar Katchi_Rank', 'Desiya Murpokku Dravida Kazhagam_Rank',
       'Tamilaga Vettri Kazhagam_Rank', 'Puthiya Tamilagam_Rank',
       'Nam Naadu Nam Makkal Nam Ethirkaa

In [5]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the eighth dataset (Update the filename to match your file)
df8 = pd.read_csv("Detailed_Polling_Data.csv")

# 2. Select the major political forces present in this specific index
core_parties = [
    'All India Anna Dravida Munnetra Kazhagam', 
    'Desiya Murpokku Dravida Kazhagam',
    'Naam Tamilar Katchi', 
    'Tamilaga Vettri Kazhagam'
]

# Clean missing numerical fields by filling with 0
df8[core_parties] = df8[core_parties].fillna(0)

# 3. Calculate true total votes for normalization (Core + Independents + NOTA)
df8['Total_Calculated_Votes'] = df8[core_parties].sum(axis=1) + df8['Total_Independent_Votes'].fillna(0) + df8['NOTA'].fillna(0)

# Filter out empty entries to completely avoid division by zero errors
df8 = df8[df8['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    col_name = f'{party}_share_pct'
    df8[col_name] = (df8[party] / df8['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Append strategic structural dimensions
df8['independent_share_pct'] = (df8['Total_Independent_Votes'].fillna(0) / df8['Total_Calculated_Votes']) * 100
feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']

# Drop or fill edge-case missing numbers inside target features
df8[feature_cols] = df8[feature_cols].fillna(0)

# 5. Extract and Scale features for the ML model
X = df8[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df8['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown to help map the text identities
print("\n--- DATASET 8: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df8.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 8: BOOTH COUNT PER CLUSTER ---")
print(df8['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    cluster_df = df8[df8['Cluster_ID'] == cluster_num][
        [
            'Serial No. Of Polling Station', 
            'Location and name of Building in Which Polling Station Located', 
            'Polling Area', 
            'Winner_Party', 
            'Margin_Percentage'
        ]
    ]
    filename = f"Dataset_8_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")



--- DATASET 8: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            All India Anna Dravida Munnetra Kazhagam_share_pct  \
Cluster_ID                                                       
0                                                       28.13    
1                                                       19.52    
2                                                       19.56    
3                                                       18.11    

            Desiya Murpokku Dravida Kazhagam_share_pct  \
Cluster_ID                                               
0                                                32.44   
1                                                50.17   
2                                                35.81   
3                                                26.24   

            Naam Tamilar Katchi_share_pct  Tamilaga Vettri Kazhagam_share_pct  \
Cluster_ID                                                                      
0                              